# LeetCode 2815 — Max Pair Sum in an Array bằng Python

Link đề: https://leetcode.com/problems/max-pair-sum-in-an-array/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Math

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def maxSum(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong mảng `nums`, tìm **hai phần tử** (ở hai vị trí khác nhau) có **chữ số lớn nhất** (max digit) **bằng nhau** sao cho **tổng** của chúng lớn nhất; không có cặp nào thì trả `-1`.
2. Biết **tự tính tay**: với mỗi số, tìm chữ số lớn nhất của nó, nhóm các số theo chữ số đó, rồi chọn hai số lớn nhất trong mỗi nhóm.
3. Học các công cụ Python: `str(x)`, `max` trên chuỗi, `%` và `//` để bóc chữ số, `dict`, `defaultdict(list)`, `sorted`, `-1` làm giá trị "không có".
4. Đi từ **Brute Force kiểm mọi cặp (`O(n²·d)`)** → **gom nhóm theo chữ số lớn nhất** → **một lượt với mảng `best[10]`** → **sắp xếp giảm dần**.
5. Chứng minh vì sao chỉ cần quan tâm **hai số lớn nhất mỗi nhóm**, và tránh lỗi kinh điển: dùng cùng một phần tử hai lần, quên trả `-1`, nhầm "chữ số lớn nhất" với "giá trị lớn nhất".

## 2. Đề bài & ví dụ

Cho một mảng số nguyên `nums`. Hãy tìm **tổng lớn nhất** của một cặp số `nums[i]` và `nums[j]` (với `i != j`, và thường lấy `i < j`) sao cho **chữ số lớn nhất** của `nums[i]` **bằng** chữ số lớn nhất của `nums[j]`.

Trả về tổng lớn nhất tìm được, hoặc `-1` nếu không tồn tại cặp nào thoả mãn.

Ví dụ chữ số lớn nhất: số `2536` có các chữ số `2, 5, 3, 6` ⇒ chữ số lớn nhất là `6`; số `51` có chữ số lớn nhất là `5`.

| Ví dụ | `nums` | Output | Giải thích |
|---|---|---|---|
| 1 | `[51, 71, 17, 24, 42]` | `88` | `51` → 5, `71` → 7, `17` → 7, `24` → 4, `42` → 4. Cặp `(71, 17)` cùng 7, tổng 88. Cặp `(24, 42)` cùng 4, tổng 66. Lớn nhất là 88. |
| 2 | `[1, 2, 3, 4]` | `-1` | Chữ số lớn nhất của mỗi số là chính nó, đều khác nhau ⇒ không có cặp nào. |

Constraints:

- `2 <= nums.length <= 100`
- `1 <= nums[i] <= 10^4`

In [1]:
# Chạy thử "bằng tay" ví dụ 1: tìm chữ số lớn nhất của từng số
nums = [51, 71, 17, 24, 42]
for x in nums:
    print(x, "-> chữ số lớn nhất:", max(str(x)))

51 -> chữ số lớn nhất: 5
71 -> chữ số lớn nhất: 7
17 -> chữ số lớn nhất: 7
24 -> chữ số lớn nhất: 4
42 -> chữ số lớn nhất: 4


## 3. Kiến thức nền

### 3.1. Bóc chữ số bằng `str`

Chuyển số sang chuỗi bằng `str(x)`, rồi duyệt từng ký tự. `max("2536")` so sánh ký tự theo bảng mã; vì `'0' < '1' < ... < '9'` nên `max` của chuỗi chữ số chính là **chữ số lớn nhất** (ở dạng ký tự). Muốn có số nguyên thì `int(...)`.

In [2]:
print(str(2536))                 # '2536'
print(max(str(2536)))            # '6'
print(int(max(str(2536))))       # 6
print(list(str(2536)))           # ['2', '5', '3', '6']

2536
6
6
['2', '5', '3', '6']


### 3.2. Bóc chữ số bằng `%` và `//` (không dùng chuỗi)

`x % 10` là chữ số cuối, `x // 10` bỏ chữ số cuối. Lặp tới khi `x == 0` thì thu được mọi chữ số. Cách này cũng rất hay gặp trong các bài làm việc với chữ số.

In [3]:
x = 2536
digits = []
while x > 0:
    digits.append(x % 10)    # chữ số cuối
    x //= 10                 # bỏ chữ số cuối
print(digits)                # [6, 3, 5, 2] (từ phải sang trái)
print(max(digits))           # 6

[6, 3, 5, 2]
6


### 3.3. `dict` và `defaultdict(list)` để nhóm

`dict` lưu cặp khóa → giá trị. Muốn **nhóm** các số theo chữ số lớn nhất, ta dùng `d[chữ_số].append(số)`. `defaultdict(list)` tự tạo list rỗng khi khóa chưa có, nên khỏi phải kiểm tra `if key not in d`.

In [4]:
from collections import defaultdict

groups = defaultdict(list)
for x in [51, 71, 17, 24, 42]:
    groups[max(str(x))].append(x)
print(dict(groups))          # {'5': [51], '7': [71, 17], '4': [24, 42]}

{'5': [51], '7': [71, 17], '4': [24, 42]}


### 3.4. `sorted(..., reverse=True)` và lấy hai phần tử lớn nhất

`sorted(lst, reverse=True)` trả list mới **giảm dần**; hai phần tử lớn nhất là `lst2[0]` và `lst2[1]`. Ta chỉ lấy khi list có ít nhất 2 phần tử (`len(lst) >= 2`).

In [5]:
lst = [71, 17]
lst2 = sorted(lst, reverse=True)
print(lst2, lst2[0] + lst2[1])   # [71, 17] 88
print(sorted([5, 9, 1, 9], reverse=True)[:2])   # [9, 9]

[71, 17] 88
[9, 9]


### 3.5. Giá trị "chưa có" và `-1`

Khi chưa tìm thấy cặp nào, ta trả `-1`. Một mẹo: khởi tạo `ans = -1`, và mỗi lần tìm được tổng hợp lệ ta làm `ans = max(ans, tong)`. Vì mọi tổng hợp lệ `>= 2` (do `nums[i] >= 1`), nên `-1` không bao giờ "thắng" nhầm.

In [6]:
ans = -1
for tong in [66, 88]:
    ans = max(ans, tong)
print(ans)       # 88
ans = -1
print(ans)       # -1  (không có cặp)

88
-1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 1 `[51, 71, 17, 24, 42]`.

**Bước 1 — tìm chữ số lớn nhất của từng số:**

| số | các chữ số | chữ số lớn nhất |
|---|---|---|
| 51 | 5, 1 | 5 |
| 71 | 7, 1 | 7 |
| 17 | 1, 7 | 7 |
| 24 | 2, 4 | 4 |
| 42 | 4, 2 | 4 |

**Bước 2 — nhóm theo chữ số lớn nhất:** nhóm 5 = {51}; nhóm 7 = {71, 17}; nhóm 4 = {24, 42}.

**Bước 3 — mỗi nhóm có ít nhất 2 số mới tạo được cặp.** Nhóm 5 chỉ có 1 số ⇒ bỏ. Nhóm 7: 71 + 17 = 88. Nhóm 4: 24 + 42 = 66.

**Bước 4 — lấy lớn nhất:** 88.

**Nhận xét:** trong **mỗi nhóm**, tổng lớn nhất luôn là **hai số lớn nhất** của nhóm đó. Việc còn lại là cài đặt hiệu quả.

In [7]:
nums = [51, 71, 17, 24, 42]
nhom = {}
for x in nums:
    nhom.setdefault(max(str(x)), []).append(x)
for d, lst in nhom.items():
    if len(lst) >= 2:
        top2 = sorted(lst, reverse=True)[:2]
        print(f"nhóm {d}: {lst} -> hai số lớn nhất {top2} tổng {sum(top2)}")
    else:
        print(f"nhóm {d}: {lst} -> chưa đủ 2 số")

nhóm 5: [51] -> chưa đủ 2 số
nhóm 7: [71, 17] -> hai số lớn nhất [71, 17] tổng 88
nhóm 4: [24, 42] -> hai số lớn nhất [42, 24] tổng 66


## 5. Cách 1 — Brute Force / baseline: kiểm mọi cặp

**Ý tưởng:** duyệt mọi cặp `(i, j)` với `i < j`; nếu chữ số lớn nhất của `nums[i]` bằng của `nums[j]` thì thử cập nhật đáp án bằng `nums[i] + nums[j]`.

Để khỏi tính lại chữ số lớn nhất nhiều lần, ta viết hàm phụ `max_digit(x)`.

In [8]:
from typing import List

def max_digit(x: int) -> int:
    """Chữ số lớn nhất của x (x >= 1)."""
    best = 0
    while x > 0:
        best = max(best, x % 10)
        x //= 10
    return best

def max_sum_bruteforce(nums: List[int]) -> int:
    n = len(nums)
    ans = -1
    for i in range(n):
        for j in range(i + 1, n):
            if max_digit(nums[i]) == max_digit(nums[j]):
                ans = max(ans, nums[i] + nums[j])
    return ans

In [9]:
print(max_digit(2536), max_digit(51), max_digit(100))   # 6 5 1
print(max_sum_bruteforce([51, 71, 17, 24, 42]))          # 88
print(max_sum_bruteforce([1, 2, 3, 4]))                  # -1

6 5 1
88
-1


**Độ phức tạp:** gọi `n = len(nums)`, `d` là số chữ số tối đa (ở đây `d <= 5`). Có `n(n−1)/2` cặp, mỗi cặp tính hai `max_digit` tốn `O(d)` ⇒ Time `O(n²·d)`. Space `O(1)`.

**Nút thắt:** ta tính đi tính lại `max_digit` của cùng một số cho nhiều cặp. Và so sánh từng cặp là thừa: chỉ cần biết **hai số lớn nhất của từng nhóm**.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — tính `max_digit` một lần cho mỗi số.** Mỗi số chỉ thuộc **một nhóm** (nhóm theo chữ số lớn nhất, có tối đa 9 nhóm vì chữ số lớn nhất là `1..9` — `0` chỉ xuất hiện khi số bằng 0, mà `nums[i] >= 1`).

**Đột phá 2 — trong mỗi nhóm chỉ cần hai số lớn nhất.** Giả sử nhóm có các số `a ≥ b ≥ c ≥ ...`. Tổng hai số bất kỳ không vượt quá `a + b` (vì thay một số bằng số lớn hơn thì tổng không giảm). Nên tổng tốt nhất của nhóm là `a + b`.

**Đột phá 3 — có thể làm một lượt (one pass).** Duyệt từng `x` và giữ mảng `best[d]` = số lớn nhất **đã gặp** có chữ số lớn nhất `d`. Với `x` mới:

1. `d = max_digit(x)`.
2. Nếu `best[d]` tồn tại, thì `best[d] + x` là một tổng hợp lệ: cập nhật `ans`.
3. Cập nhật `best[d] = max(best[d], x)`.

Vì ta luôn ghép `x` với số **lớn nhất trước đó** của cùng nhóm, nên mọi cặp tối ưu đều được xét (cặp tối ưu `(a, b)` của một nhóm sẽ được tính khi gặp phần tử đến sau trong hai phần tử).

In [10]:
best = {}          # chữ số lớn nhất -> số lớn nhất đã gặp trong nhóm đó
ans = -1
for x in [51, 71, 17, 24, 42]:
    d = max_digit(x)
    if d in best:
        ans = max(ans, best[d] + x)
    best[d] = max(best.get(d, 0), x)
    print(f"x={x:3} d={d} best={best} ans={ans}")

x= 51 d=5 best={5: 51} ans=-1
x= 71 d=7 best={5: 51, 7: 71} ans=-1
x= 17 d=7 best={5: 51, 7: 71} ans=88
x= 24 d=4 best={5: 51, 7: 71, 4: 24} ans=88
x= 42 d=4 best={5: 51, 7: 71, 4: 42} ans=88


## 7. Các cách tiếp theo

Ta giảng thêm 3 cách nữa.

### Cách 2 — Gom nhóm bằng `defaultdict`, lấy hai số lớn nhất mỗi nhóm

**Từng đoạn:**

- Duyệt `nums`, `groups[max_digit(x)].append(x)`.
- Với mỗi nhóm có `>= 2` phần tử: sắp xếp giảm dần, lấy tổng hai phần tử đầu.
- Trả về tổng lớn nhất qua các nhóm, hoặc `-1`.

In [11]:
from collections import defaultdict

def max_sum_groups(nums: List[int]) -> int:
    groups = defaultdict(list)
    for x in nums:
        groups[max_digit(x)].append(x)
    ans = -1
    for lst in groups.values():
        if len(lst) >= 2:
            lst.sort(reverse=True)
            ans = max(ans, lst[0] + lst[1])
    return ans

In [12]:
print(max_sum_groups([51, 71, 17, 24, 42]))   # 88
print(max_sum_groups([1, 2, 3, 4]))           # -1

88
-1


**Code nộp LeetCode (Cách 2):**

```python
from typing import List
from collections import defaultdict

class Solution:
    def maxSum(self, nums: List[int]) -> int:
        groups = defaultdict(list)
        for x in nums:
            groups[max(str(x))].append(x)
        ans = -1
        for lst in groups.values():
            if len(lst) >= 2:
                lst.sort(reverse=True)
                ans = max(ans, lst[0] + lst[1])
        return ans
```

**Độ phức tạp:** Time `O(n·d + n log n)` (tính chữ số + sắp xếp các nhóm), Space `O(n)`.

### Cách 3 — Một lượt với mảng `best[10]` (tối ưu)

**Từng đoạn:**

- `best = [0] * 10`: `best[d] = 0` nghĩa là "chưa có số nào" (vì `nums[i] >= 1`, số thật luôn `>= 1`).
- Với mỗi `x`: `d = max_digit(x)`; nếu `best[d] > 0` thì `ans = max(ans, best[d] + x)`; rồi `best[d] = max(best[d], x)`.
- Không cần dict hay sắp xếp, bộ nhớ chỉ 10 ô.

In [13]:
def max_sum_onepass(nums: List[int]) -> int:
    best = [0] * 10
    ans = -1
    for x in nums:
        d = max_digit(x)
        if best[d] > 0:
            ans = max(ans, best[d] + x)
        best[d] = max(best[d], x)
    return ans

In [14]:
print(max_sum_onepass([51, 71, 17, 24, 42]))   # 88
print(max_sum_onepass([1, 2, 3, 4]))           # -1
print(max_sum_onepass([2536, 1613, 3366, 162]))  # tự tính: cả bốn số đều có chữ số lớn nhất là 6

88
-1
5902


**Code nộp LeetCode (Cách 3):**

```python
from typing import List

class Solution:
    def maxSum(self, nums: List[int]) -> int:
        best = [0] * 10
        ans = -1
        for x in nums:
            d = int(max(str(x)))
            if best[d] > 0:
                ans = max(ans, best[d] + x)
            best[d] = max(best[d], x)
        return ans
```

**Độ phức tạp:** Time `O(n·d)`, Space `O(1)` (mảng 10 ô cố định).

### Cách 4 — Sắp xếp giảm dần rồi chọn cặp đầu tiên của mỗi nhóm

**Từng đoạn:**

- Sắp xếp `nums` giảm dần.
- Duyệt theo thứ tự đó, với mỗi chữ số `d` ta lưu số **đầu tiên** gặp (đó là số lớn nhất của nhóm). Khi gặp số thứ hai cùng nhóm, đó là số lớn thứ hai ⇒ tổng của nhóm.
- Vì mỗi nhóm chỉ tính tổng ở lần gặp thứ hai nên không cần xét thêm.

In [15]:
def max_sum_sorted(nums: List[int]) -> int:
    first = {}               # chữ số lớn nhất -> số lớn nhất của nhóm
    ans = -1
    for x in sorted(nums, reverse=True):
        d = max_digit(x)
        if d in first:
            if first[d] is not None:
                ans = max(ans, first[d] + x)
                first[d] = None          # đã chốt cặp của nhóm này
        else:
            first[d] = x
    return ans

In [16]:
print(max_sum_sorted([51, 71, 17, 24, 42]))   # 88
print(max_sum_sorted([1, 2, 3, 4]))           # -1

88
-1


**Code nộp LeetCode (Cách 4)** có cùng thân hàm như trên trong class `Solution`.

**Độ phức tạp:** sắp xếp `O(n log n)` + duyệt `O(n·d)` ⇒ Time `O(n log n + n·d)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 (một lượt với `best[10]`) trên `nums = [51, 71, 17, 24, 42]`.

| `x` | `d = max_digit(x)` | `best[d]` trước | tổng thử `best[d] + x` | `ans` sau | `best[d]` sau |
|---|---|---|---|---|---|
| 51 | 5 | 0 | — | −1 | 51 |
| 71 | 7 | 0 | — | −1 | 71 |
| 17 | 7 | 71 | 88 | **88** | 71 |
| 24 | 4 | 0 | — | 88 | 24 |
| 42 | 4 | 24 | 66 | 88 | 42 |

In [17]:
def trace_onepass(nums):
    best = [0] * 10
    ans = -1
    for x in nums:
        d = max_digit(x)
        truoc = best[d]
        if truoc > 0:
            tong = truoc + x
            ans = max(ans, tong)
            ghi_chu = f"thử tổng {truoc} + {x} = {tong}"
        else:
            ghi_chu = "chưa có số cùng nhóm"
        best[d] = max(best[d], x)
        print(f"x={x:5} d={d} best[d] trước={truoc:5} | {ghi_chu:28} | ans={ans:5} | best[d] sau={best[d]}")
    return ans

print(trace_onepass([51, 71, 17, 24, 42]))

x=   51 d=5 best[d] trước=    0 | chưa có số cùng nhóm         | ans=   -1 | best[d] sau=51
x=   71 d=7 best[d] trước=    0 | chưa có số cùng nhóm         | ans=   -1 | best[d] sau=71
x=   17 d=7 best[d] trước=   71 | thử tổng 71 + 17 = 88        | ans=   88 | best[d] sau=71
x=   24 d=4 best[d] trước=    0 | chưa có số cùng nhóm         | ans=   88 | best[d] sau=24
x=   42 d=4 best[d] trước=   24 | thử tổng 24 + 42 = 66        | ans=   88 | best[d] sau=42
88


In [18]:
# Một ví dụ mà số đến sau lại LỚN hơn số trước trong cùng nhóm: [17, 71, 27]
print(trace_onepass([17, 71, 27]))

x=   17 d=7 best[d] trước=    0 | chưa có số cùng nhóm         | ans=   -1 | best[d] sau=17
x=   71 d=7 best[d] trước=   17 | thử tổng 17 + 71 = 88        | ans=   88 | best[d] sau=71
x=   27 d=7 best[d] trước=   71 | thử tổng 71 + 27 = 98        | ans=   98 | best[d] sau=71
98


Ở ví dụ thứ hai, `17` và `71` cùng nhóm 7 (tổng 88), `27` cũng nhóm 7. `best[7]` sau hai số đầu là `71`, nên `27` được ghép với `71` cho 98, lớn hơn 88. Điều này cho thấy ta phải giữ số **lớn nhất** của nhóm, không phải số gần nhất.

## 9. Code nộp LeetCode

Dùng Cách 3 (một lượt, `best[10]`): gọn, bộ nhớ hằng số.

In [19]:
from typing import List

class Solution:
    def maxSum(self, nums: List[int]) -> int:
        best = [0] * 10          # best[d] = số lớn nhất đã gặp có chữ số lớn nhất là d
        ans = -1
        for x in nums:
            d = int(max(str(x)))
            if best[d] > 0:
                ans = max(ans, best[d] + x)
            best[d] = max(best[d], x)
        return ans

In [20]:
print(Solution().maxSum([51, 71, 17, 24, 42]))   # 88
print(Solution().maxSum([1, 2, 3, 4]))           # -1

88
-1


## 10. Vì sao thuật toán đúng

**Gọi** nhóm `G_d` là tập các vị trí có chữ số lớn nhất bằng `d`. Đáp án đúng là `max` trên các `d` có `|G_d| >= 2` của tổng hai phần tử **lớn nhất** trong `G_d`.

**Bước 1 — tổng tốt nhất trong một nhóm là hai số lớn nhất.** Cho cặp bất kỳ `(u, v)` trong nhóm và `a ≥ b` là hai số lớn nhất của nhóm (ở hai vị trí khác nhau). Khi đó `u ≤ a` và `v ≤ a`; hơn nữa vì `u` và `v` ở hai vị trí khác nhau nên số nhỏ hơn trong `u, v` không vượt quá `b`. Do đó `u + v ≤ a + b`.

**Bước 2 — cách một lượt tính đúng `a + b` của mỗi nhóm.** Bất biến: trước khi xử lý phần tử `x` thuộc nhóm `d`, `best[d]` là số lớn nhất của nhóm `d` **trong các phần tử đã duyệt**. Gọi `a ≥ b` là hai số lớn nhất của nhóm (ở hai vị trí khác nhau). Trong hai vị trí đó, gọi `y` là phần tử **đến sau** và `z` là phần tử **đến trước**. Lúc xử lý `y`, `z` đã được duyệt nên `best[d] ≥ z`, do đó tổng thử `best[d] + y ≥ z + y = a + b`. Mặt khác tổng thử luôn là tổng của hai phần tử ở hai vị trí khác nhau trong nhóm nên `≤ a + b` (Bước 1). Vậy tổng thử đúng bằng `a + b`, và `ans` ghi nhận được nó.

**Bước 3 — `-1`.** Nếu không nhóm nào có hai phần tử thì `best[d] > 0` không bao giờ xảy ra tại lần gặp thứ hai, `ans` giữ `-1`.

**Bước 4 — dùng `0` làm "chưa có".** Vì `nums[i] >= 1`, một số thật luôn `> 0`, nên `best[d] > 0` ⇔ nhóm `d` đã có số.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `d` = số chữ số tối đa của một số (`d <= 5`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute Force mọi cặp | `O(n²·d)` | `O(1)` | oracle |
| 2. Gom nhóm + sort mỗi nhóm | `O(n·d + n log n)` | `O(n)` | dễ hiểu |
| 3. Một lượt `best[10]` | `O(n·d)` | `O(1)` | **code nộp** |
| 4. Sort giảm dần + nhóm | `O(n log n + n·d)` | `O(n)` | thay cho cách 2 |

Với `n <= 100`, `nums[i] <= 10^4`, mọi cách nhanh. Cách 3 gọn và có bộ nhớ hằng.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: không có cặp (`-1`); đúng hai phần tử cùng nhóm; hai phần tử khác nhóm; phần tử trùng giá trị (hai số bằng nhau cũng là hai phần tử khác vị trí); nhóm có nhiều hơn hai số; số có chữ số `0`.

In [21]:
impls = {
    "bruteforce": max_sum_bruteforce,
    "groups": max_sum_groups,
    "onepass": max_sum_onepass,
    "sorted": max_sum_sorted,
    "Solution": Solution().maxSum,
}
cases = [
    ("ví dụ 1", [51, 71, 17, 24, 42], 88),
    ("ví dụ 2", [1, 2, 3, 4], -1),
    ("hai số cùng nhóm", [12, 21], 33),
    ("hai số khác nhóm", [12, 13], -1),
    ("trùng giá trị", [5, 5], 10),
    ("ba số cùng nhóm lấy hai lớn nhất", [19, 91, 29], 120),
    ("chữ số 0 không làm hỏng", [100, 10, 1], 110),
    ("max digit khác giá trị lớn nhất", [112, 131, 411], -1),
    ("bốn số cùng max digit 6", [2536, 1613, 3366, 162], 5902),
]
for name, nums, exp in cases:
    for k, f in impls.items():
        got = f(list(nums))
        assert got == exp, (name, k, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 88
PASS ví dụ 2 -> -1
PASS hai số cùng nhóm -> 33
PASS hai số khác nhóm -> -1
PASS trùng giá trị -> 10
PASS ba số cùng nhóm lấy hai lớn nhất -> 120
PASS chữ số 0 không làm hỏng -> 110
PASS max digit khác giá trị lớn nhất -> -1
PASS bốn số cùng max digit 6 -> 5902


Kiểm tay vài ca: `[100, 10, 1]` đều có chữ số lớn nhất `1`, hai số lớn nhất `100 + 10 = 110` ✓. `[19, 91, 29]` đều có max digit `9`; hai số lớn nhất `91 + 29 = 120` ✓. `[112, 131, 411]`: max digit lần lượt `2, 3, 4` ⇒ `-1` ✓. `[2536, 1613, 3366, 162]`: cả bốn có max digit `6`; hai số lớn nhất `3366 + 2536 = 5902` ✓.

### 12.2. So sánh các cách trên nhiều input ngẫu nhiên có seed

Input hợp lệ theo constraints: `2 <= n <= 100`, `1 <= nums[i] <= 10^4`. Ta trộn nhiều dải giá trị để có lúc nhiều cặp, lúc ít.

In [22]:
import random
random.seed(2815)
for _ in range(1500):
    n = random.randint(2, 100)
    hi = random.choice([9, 99, 999, 10000])
    nums = [random.randint(1, hi) for _ in range(n)]
    exp = max_sum_bruteforce(nums)
    for k, f in impls.items():
        assert f(list(nums)) == exp, (nums, k)
print("1500 mảng ngẫu nhiên: mọi cách khớp brute force")

1500 mảng ngẫu nhiên: mọi cách khớp brute force


### 12.3. Kiểm tính chất

- Đáp án là `-1` hoặc một tổng hai phần tử khác vị trí của `nums`.
- Hoán vị `nums` không đổi đáp án.
- Thêm một phần tử vào mảng không làm đáp án **giảm**.

In [23]:
random.seed(28150)
for _ in range(500):
    n = random.randint(2, 30)
    nums = [random.randint(1, 10000) for _ in range(n)]
    r = Solution().maxSum(list(nums))
    if r != -1:
        ok = any(nums[i] + nums[j] == r for i in range(n) for j in range(i + 1, n))
        assert ok
    shuffled = nums[:]
    random.shuffle(shuffled)
    assert Solution().maxSum(shuffled) == r
    extra = nums + [random.randint(1, 10000)]
    assert Solution().maxSum(extra) >= r
print("tính chất đúng")

tính chất đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng cùng một phần tử hai lần

In [24]:
def sai_dung_hai_lan(nums):
    best = [0] * 10
    ans = -1
    for x in nums:
        d = max_digit(x)
        best[d] = max(best[d], x)          # sai: cập nhật TRƯỚC khi ghép
        if best[d] > 0:
            ans = max(ans, best[d] + x)    # ghép x với chính x
    return ans

print("sai :", sai_dung_hai_lan([1, 2, 3, 4]))      # 8 (ghép 4 với 4)
print("đúng:", Solution().maxSum([1, 2, 3, 4]))     # -1

sai : 8
đúng: -1


**Vì sao sai:** nếu cập nhật `best[d]` trước khi ghép, `x` sẽ ghép với chính nó. Phải **ghép trước, cập nhật sau**.

### Lỗi 2: Nhầm "chữ số lớn nhất" với "giá trị lớn nhất"

In [25]:
def sai_nham_gia_tri(nums):
    top2 = sorted(nums)[-2:]          # sai: lấy hai số lớn nhất của cả mảng
    return sum(top2)

print("sai :", sai_nham_gia_tri([1, 2, 3, 4]))       # 7
print("đúng:", Solution().maxSum([1, 2, 3, 4]))      # -1

sai : 7
đúng: -1


**Vì sao sai:** đề yêu cầu **chữ số lớn nhất của hai số bằng nhau**, không phải chọn hai số lớn nhất của cả mảng. Phải tính `max_digit` rồi so sánh.

### Lỗi 3: Quên trả `-1` (trả `0` hoặc tổng sai)

In [26]:
def sai_tra_0(nums):
    best = [0] * 10
    ans = 0                      # sai: nên là -1
    for x in nums:
        d = max_digit(x)
        if best[d] > 0:
            ans = max(ans, best[d] + x)
        best[d] = max(best[d], x)
    return ans

print("sai :", sai_tra_0([1, 2, 3, 4]))
print("đúng:", Solution().maxSum([1, 2, 3, 4]))

sai : 0
đúng: -1


**Vì sao sai:** đề nói không có cặp thì trả `-1`. Khởi tạo `ans = -1` (vì mọi tổng hợp lệ `>= 2`, nên không bị đè sai).

### Lỗi 4: So sánh ký tự và số

In [27]:
d1 = max(str(51))      # '5' (chuỗi)
d2 = 5                 # số nguyên
print(d1 == d2)        # False !
print(int(d1) == d2)   # True

False
True


**Vì sao sai:** `max(str(x))` trả **ký tự** `'5'`, không phải số `5`. Dùng làm chỉ số mảng `best['5']` sẽ lỗi `TypeError`; hãy bọc `int(...)` (hoặc dùng dict với khoá ký tự đồng nhất).

### Lỗi 5 (nhỏ): Chữ số 0 làm `max_digit` sai khi số là bội của 10

`max_digit(100) = 1` (chữ số lớn nhất là `1`, không phải `0`). Nếu bạn khởi tạo `best_digit = x % 10` rồi mới so sánh, số `100` sẽ cho `0` và bị xếp vào nhóm `0`.

In [28]:
def sai_max_digit(x):
    return x % 10           # sai: chỉ lấy chữ số cuối

print(sai_max_digit(100), max_digit(100))     # 0 1

0 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "tìm cặp có **tính chất chung** (cùng khoá) và tổng lớn nhất" ⇒ **nhóm theo khoá**, mỗi nhóm giữ **số lớn nhất** (hoặc hai số lớn nhất).
- Mẹo one-pass: **ghép với `best[khoá]` trước, rồi cập nhật `best[khoá]`** để không ghép phần tử với chính nó.
- Khoá chỉ có ít giá trị (chữ số `0..9`) ⇒ dùng **mảng** thay dict cho gọn.
- Bóc chữ số: `max(str(x))` cho code ngắn, `% 10` / `// 10` cho vòng lặp thuần số.
- Khi đề yêu cầu `-1` nếu không có đáp án: khởi tạo `ans = -1` và chỉ cập nhật bằng `max`.

## 15. Biến thể của bài

1. **Cùng chữ số nhỏ nhất:** thay `max_digit` bằng chữ số nhỏ nhất.
2. **Cùng tổng các chữ số** (xem LeetCode 2342): nhóm theo tổng chữ số, lấy tổng lớn nhất của hai số.
3. **Trả về chỉ số `(i, j)`** thay vì tổng: lưu thêm vị trí vào `best`.
4. **Lấy `k` số lớn nhất** mỗi nhóm: dùng `heapq` kích thước `k`.
5. **Tích lớn nhất** thay vì tổng (cẩn thận số âm nếu mở rộng).
6. **`n` rất lớn (`10^6`):** cách one-pass vẫn `O(n·d)`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute Force | mọi cặp, so `max_digit` | `O(n²·d)` | `O(1)` | oracle, `n` nhỏ |
| 2. Gom nhóm + sort | dict nhóm, hai số lớn nhất | `O(n·d + n log n)` | `O(n)` | dễ hiểu nhất |
| 3. Một lượt `best[10]` | ghép với số lớn nhất trước rồi cập nhật | `O(n·d)` | `O(1)` | **code nộp**, `n` lớn |
| 4. Sort giảm dần | số đầu tiên mỗi nhóm là lớn nhất | `O(n log n + n·d)` | `O(n)` | thích dùng `sorted` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `max(str(4072))`, `4072 % 10`, `4072 // 10`, `max(0, -1)`.

**Bài 2 (tính tay).** Với `nums = [31, 13, 25, 52, 7]`, lập bảng chữ số lớn nhất, các nhóm và đáp án.

**Bài 3 (điền TODO).** Hoàn thành `max_sum_todo(nums)` theo Cách 3.

In [29]:
def max_sum_todo(nums):
    # TODO 1: best = [0] * 10, ans = -1
    # TODO 2: duyệt từng x
    # TODO 3: d = chữ số lớn nhất của x
    # TODO 4: nếu best[d] > 0 thì ans = max(ans, best[d] + x)
    # TODO 5: best[d] = max(best[d], x)
    # TODO 6: return ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `max_sum_groups2(nums)` bằng `defaultdict(list)` **không nhìn** code mẫu. Thử với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `max_pair_indices(nums)` trả về cặp chỉ số `(i, j)`, `i < j`, cho tổng lớn nhất (hoặc `None`). (b) Viết `max_sum_same_digit_sum(nums)`: tìm tổng lớn nhất của hai số có **cùng tổng các chữ số** (không có thì `-1`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `max(str(4072)) = '7'`, `4072 % 10 = 2`, `4072 // 10 = 407`. Bài 2: tìm chữ số lớn nhất từng số: 31→3, 13→3, 25→5, 52→5, 7→7.
- **Mức 2 (vừa):** Bài 3: sáu dòng đã ghi sẵn trong comment TODO. Bài 4: `groups[max_digit(x)].append(x)` rồi sort mỗi nhóm giảm dần.
- **Mức 3 (sâu):** Bài 5(a): lưu `best[d] = (giá_trị, chỉ_số)`; khi gặp `x` ở chỉ số `j`, nếu `best[d]` có thì cặp là `(best_index, j)` (đã `< j`). Bài 5(b): thay `max_digit` bằng `digit_sum` và đổi mảng 10 ô thành dict (tổng chữ số tới 36).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert max(str(4072)) == "7"
assert 4072 % 10 == 2 and 4072 // 10 == 407
assert max(0, -1) == 0

# Bài 2: [31, 13, 25, 52, 7]
# nhóm 3 = {31, 13} -> 44 ; nhóm 5 = {25, 52} -> 77 ; nhóm 7 = {7} -> không đủ
assert Solution().maxSum([31, 13, 25, 52, 7]) == 77

# Bài 3
def max_sum_todo_ans(nums):
    best = [0] * 10
    ans = -1
    for x in nums:
        d = max_digit(x)
        if best[d] > 0:
            ans = max(ans, best[d] + x)
        best[d] = max(best[d], x)
    return ans

assert max_sum_todo_ans([51, 71, 17, 24, 42]) == 88
assert max_sum_todo_ans([1, 2, 3, 4]) == -1

# Bài 4
def max_sum_groups2(nums):
    groups = defaultdict(list)
    for x in nums:
        groups[max_digit(x)].append(x)
    ans = -1
    for lst in groups.values():
        if len(lst) >= 2:
            lst.sort(reverse=True)
            ans = max(ans, lst[0] + lst[1])
    return ans

assert max_sum_groups2([51, 71, 17, 24, 42]) == 88
assert max_sum_groups2([1, 2, 3, 4]) == -1

# Bài 5(a)
def max_pair_indices(nums):
    best = {}                    # d -> (giá trị, chỉ số) lớn nhất
    ans, ans_idx = -1, None
    for j, x in enumerate(nums):
        d = max_digit(x)
        if d in best:
            v, i = best[d]
            if v + x > ans:
                ans, ans_idx = v + x, (i, j)
        if d not in best or x > best[d][0]:
            best[d] = (x, j)
    return ans_idx

assert max_pair_indices([51, 71, 17, 24, 42]) == (1, 2)
assert max_pair_indices([1, 2, 3, 4]) is None

# Bài 5(b)
def digit_sum(x):
    return sum(int(c) for c in str(x))

def max_sum_same_digit_sum(nums):
    best = {}
    ans = -1
    for x in nums:
        s = digit_sum(x)
        if s in best:
            ans = max(ans, best[s] + x)
        best[s] = max(best.get(s, 0), x)
    return ans

assert max_sum_same_digit_sum([18, 43, 36, 13, 7]) == 54    # 18 và 36 cùng tổng chữ số 9
assert max_sum_same_digit_sum([10, 12, 19, 14]) == -1

random.seed(5)
for _ in range(300):
    nums = [random.randint(1, 10000) for _ in range(random.randint(2, 30))]
    r = max_sum_bruteforce(nums)
    assert max_sum_todo_ans(nums) == max_sum_groups2(nums) == r
    idx = max_pair_indices(nums)
    if r == -1:
        assert idx is None
    else:
        assert nums[idx[0]] + nums[idx[1]] == r and idx[0] < idx[1]
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (nhóm theo khoá, ghép với số lớn nhất của nhóm trước khi cập nhật)

```python
from typing import List

class Solution:
    def maxSum(self, nums: List[int]) -> int:
        best = [0] * 10                    # best[d] = số lớn nhất của nhóm d
        ans = -1
        for x in nums:
            d = int(max(str(x)))           # chữ số lớn nhất của x
            if best[d] > 0:
                ans = max(ans, best[d] + x)   # ghép TRƯỚC
            best[d] = max(best[d], x)         # cập nhật SAU
        return ans
```

**Checklist**

- [ ] Hiểu "chữ số lớn nhất" ≠ "giá trị lớn nhất"?
- [ ] Ghép **trước**, cập nhật `best` **sau** (không dùng một phần tử hai lần)?
- [ ] Giữ số **lớn nhất** của nhóm (không phải số gần nhất)?
- [ ] Khởi tạo `ans = -1` và trả `-1` khi không có cặp?
- [ ] Bọc `int(...)` khi lấy chữ số từ chuỗi làm chỉ số?
- [ ] Đã thử: không có cặp, hai số bằng nhau, nhóm ≥ 3 số, số có chữ số `0`?
- [ ] Đã so với brute force trên input ngẫu nhiên có seed?
- [ ] Signature đúng `maxSum(self, nums: List[int]) -> int`?